# 05. Machine Learning Experimentation & Model Comparison: Heart Disease Screening

**Phase**: Heart Disease — Phase 5  
**Document Reference**: `docs/HEART_ML_EXPERIMENTATION.md`  
**Dataset**: Cleveland Heart Disease (`datasets/raw/heart/processed.cleveland.data`)  
**Strict Phase Boundary**: Experimentation & comparison only. No clinical threshold selection, no probability calibration, no SHAP explainability, no clinical deployment.

---

### Section 1: Experiment Objective
The primary objective of this phase is to experimentally evaluate candidate machine-learning classification models for the Heart Disease binary screening task:
- $y = 0 \iff 	ext{num} = 0$ (Absence of angiographically significant CAD)
- $y = 1 \iff 	ext{num} \ge 1$ (Presence of angiographically significant CAD)

Under a rigorous, reproducible, and leakage-safe experimental protocol:
1. **Fit-on-Train Discipline**: Preprocessing is fit strictly within CV training folds; the 20% hold-out test set remains strictly isolated.
2. **Repeated Stratified Cross-Validation**: 5-repeat 5-fold CV (25 validation folds) on the 242 training samples.
3. **Model Family Exploration**: Logistic Regression (L1/L2 with standard/robust scaling), Random Forest (constrained depth and leaves), and XGBoost (shallow boosting).
4. **Multi-Metric Evaluation**: ROC-AUC, PR-AUC, Sensitivity, Specificity, Precision, F1, and Accuracy.
5. **Phase Boundaries**: This is an empirical screening benchmark, NOT a clinical deployment or threshold selection phase.


### Section 2: Environment and Dependencies
Verify Python environment, library versions, and reproducible random state configuration.


In [ ]:
import os
import sys
import hashlib
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import sklearn
import xgboost
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_curve,
    precision_recall_curve,
)

# Style configuration
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
sns.set_theme(style='whitegrid', font_scale=1.05)

RANDOM_STATE = 42

print("=== ENVIRONMENT & DEPENDENCY VERIFICATION ===")
print(f"Python:       {sys.version.split()[0]}")
print(f"NumPy:        {np.__version__}")
print(f"Pandas:       {pd.__version__}")
print(f"Scikit-Learn: {sklearn.__version__}")
print(f"XGBoost:      {xgboost.__version__}")
print(f"Random State: {RANDOM_STATE}")


### Section 3: Dataset Ingestion & Checksum Verification
Load the Cleveland Heart Disease dataset without modifying the raw storage file. Cryptographic checksums verify read-only immutability.


In [ ]:
# Add ml-service root to path if needed
ml_service_dir = os.path.abspath(os.path.join(os.getcwd(), "..", "ml-service"))
if ml_service_dir not in sys.path:
    sys.path.insert(0, ml_service_dir)

from app.heart_preprocessing import (
    verify_raw_dataset_integrity,
    load_heart_raw_data,
    create_modeling_dataset,
    split_heart_data,
    build_logistic_regression_preprocessor,
    build_tree_preprocessor,
    get_heart_feature_names,
)

valid, md5_val, sha256_val, file_size = verify_raw_dataset_integrity()
print(f"Dataset integrity verified: {valid}")
print(f"File size: {file_size} bytes")
print(f"MD5:       {md5_val}")
print(f"SHA-256:   {sha256_val}")

df_raw = load_heart_raw_data()
print(f"\nRaw dataset loaded. Shape: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns.")
df_raw.head()


### Section 4: Target Construction & Separation
Construct the binary screening target $y = (num \ge 1).astype(int)$ and separate the 13 clinical predictors $X$.


In [ ]:
X, y = create_modeling_dataset(df_raw)

print(f"Predictor matrix X shape: {X.shape}")
print(f"Target vector y shape:    {y.shape}")
print("\nTarget distribution:")
counts = y.value_counts()
pcts = y.value_counts(normalize=True) * 100
dist_df = pd.DataFrame({"Count": counts, "Percentage (%)": pcts.round(2)})
dist_df.index = ["Absence (0)", "Presence (1)"]
print(dist_df)


### Section 5: Locked Hold-Out Test Split
Partition the dataset using a stratified 80% train / 20% test split (`random_state=42`).
The 61-sample test set is locked and strictly isolated from all model exploration and CV tuning.


In [ ]:
X_train, X_test, y_train, y_test = split_heart_data(X, y, test_size=0.20, random_state=42)

print(f"Training set: {len(X_train)} samples ({y_train.value_counts()[0]} neg, {y_train.value_counts()[1]} pos)")
print(f"Holdout test: {len(X_test)} samples ({y_test.value_counts()[0]} neg, {y_test.value_counts()[1]} pos)")
print(f"Prevalence in train: {y_train.mean():.4f}")
print(f"Prevalence in test:  {y_test.mean():.4f}")
print("\n[CRITICAL]: The 61-sample test set will only be touched ONCE during final evaluation.")


### Section 6: Cross-Validation Protocol
We apply **Repeated Stratified K-Fold Cross-Validation** (5 folds, 5 repeats = 25 validation folds) strictly on the 242 training observations.
Each validation fold independently fits its own scikit-learn ColumnTransformer preprocessor to prevent data leakage.


In [ ]:
from app.heart_experimentation import (
    get_candidate_configurations,
    run_cross_validation_experiments,
    build_pipeline_from_config,
    calculate_metrics,
)

rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=42)
print(f"RSKF generated {rskf.get_n_splits(X_train, y_train)} validation folds on {len(X_train)} training observations.")


### Section 7: Logistic Regression Experiments
Evaluate 20 controlled configurations:
- Scalers: StandardScaler vs. RobustScaler
- Regularization: L1 vs. L2
- Regularization strength $C \in \{0.01, 0.1, 1.0, 10.0, 100.0\}$
- Solver: `liblinear`


In [ ]:
configs = get_candidate_configurations()
print(f"Logistic Regression search space: {len(configs['LogisticRegression'])} configurations")
for cfg in configs['LogisticRegression'][:4]:
    print(f" - {cfg['config_id']}: {cfg['description']}")
print(" ...")


### Section 8: Random Forest Experiments
Evaluate 12 controlled configurations:
- Tree count: 200, 500
- Maximum depth: None, 5, 10
- Minimum leaf size: 1, 2, 5
- Class weight: None vs. 'balanced'
- Preprocessor: Unscaled tree preprocessor


In [ ]:
print(f"Random Forest search space: {len(configs['RandomForest'])} configurations")
for cfg in configs['RandomForest'][:4]:
    print(f" - {cfg['config_id']}: {cfg['description']}")
print(" ...")


### Section 9: XGBoost Experiments
Evaluate 12 controlled configurations:
- Tree depth: 2, 3, 4
- Learning rate: 0.03, 0.05, 0.1
- Number of estimators: 100, 200, 300
- Subsample & colsample: 0.8 vs 1.0
- Preprocessor: Unscaled tree preprocessor


In [ ]:
print(f"XGBoost search space: {len(configs['XGBoost'])} configurations")
for cfg in configs['XGBoost'][:4]:
    print(f" - {cfg['config_id']}: {cfg['description']}")
print(" ...")


### Section 10: 25-Fold Cross-Validation Results
Load the pre-computed 25-fold Repeated Stratified CV results across all 44 candidate configurations.


In [ ]:
artifacts_path = os.path.join(ml_service_dir, "artifacts", "heart", "experiments")
cv_csv_path = os.path.join(artifacts_path, "cv_results_all.csv")

if os.path.exists(cv_csv_path):
    df_cv_results = pd.read_csv(cv_csv_path)
else:
    print("Running CV experiments from scratch...")
    df_cv_results, _ = run_cross_validation_experiments(X_train, y_train, random_state=42)

print(f"Cross-validation results loaded: {len(df_cv_results)} configurations evaluated across 25 folds.")
df_cv_results.sort_values(by="mean_roc_auc", ascending=False).head(10)[
    ["config_id", "family", "mean_roc_auc", "std_roc_auc", "mean_pr_auc", "std_pr_auc", "mean_f1", "mean_accuracy"]
]


### Section 11: Candidate Comparison & Multi-Metric Analysis
Compare the top performers from each model family across all primary screening metrics.


In [ ]:
for fam in ["LogisticRegression", "RandomForest", "XGBoost"]:
    top_row = df_cv_results[df_cv_results["family"] == fam].iloc[0]
    print(f"=== Best CV Configuration: {fam} ===")
    print(f"Config ID:   {top_row['config_id']}")
    print(f"ROC-AUC:     {top_row['mean_roc_auc']:.4f} ± {top_row['std_roc_auc']:.4f}")
    print(f"PR-AUC:      {top_row['mean_pr_auc']:.4f} ± {top_row['std_pr_auc']:.4f}")
    print(f"Accuracy:    {top_row['mean_accuracy']:.4f} ± {top_row['std_accuracy']:.4f}")
    print(f"Sensitivity: {top_row['mean_recall']:.4f} ± {top_row['std_recall']:.4f}")
    print(f"Specificity: {top_row['mean_specificity']:.4f} ± {top_row['std_specificity']:.4f}")
    print(f"F1-Score:    {top_row['mean_f1']:.4f} ± {top_row['std_f1']:.4f}\n")


### Section 12: Out-Of-Fold (OOF) Training Predictions
Out-of-fold predictions represent model predictions on training samples when each sample was held out in validation folds.
These will serve as the uncalibrated foundation for Phase 6 threshold analysis and Phase 7 probability calibration.


In [ ]:
oof_path = os.path.join(artifacts_path, "oof_predictions.csv")
oof_df = pd.read_csv(oof_path)
print(f"OOF predictions shape: {oof_df.shape} (242 training observations)")
oof_df.head()


### Section 13: Final Evaluation on Locked Hold-Out Test Set (N=61)
Fit each finalist candidate on the full 242-sample training set, then evaluate ONCE on the locked 61-sample test set.


In [ ]:
cand_summary_path = os.path.join(artifacts_path, "candidate_summary.csv")
df_cand_summary = pd.read_csv(cand_summary_path)
df_cand_summary[[
    "candidate_name", "test_roc_auc", "test_pr_auc", "test_accuracy",
    "test_sensitivity", "test_specificity", "test_precision", "test_f1"
]]


### Section 14: Confusion Matrices
Examine classification error patterns on the 61-sample test set under the default 0.5 threshold convention.


In [ ]:
from PIL import Image

cm_plot_path = os.path.join(artifacts_path, "confusion_matrices.png")
if os.path.exists(cm_plot_path):
    img = Image.open(cm_plot_path)
    plt.figure(figsize=(15, 5))
    plt.imshow(img)
    plt.axis("off")
    plt.show()


### Section 15: Receiver Operating Characteristic (ROC) Curves
Evaluate discrimination performance across all possible classification cutoffs.


In [ ]:
roc_plot_path = os.path.join(artifacts_path, "roc_curves.png")
if os.path.exists(roc_plot_path):
    img = Image.open(roc_plot_path)
    plt.figure(figsize=(8, 7))
    plt.imshow(img)
    plt.axis("off")
    plt.show()


### Section 16: Precision-Recall (PR) Curves
Examine trade-offs between precision (PPV) and recall (sensitivity) against the cohort prevalence baseline (45.9%).


In [ ]:
pr_plot_path = os.path.join(artifacts_path, "precision_recall_curves.png")
if os.path.exists(pr_plot_path):
    img = Image.open(pr_plot_path)
    plt.figure(figsize=(8, 7))
    plt.imshow(img)
    plt.axis("off")
    plt.show()


### Section 17: Model-Native Feature Importance Analysis
Examine model-native feature weights (L1 coefficients for Logistic Regression; split importances for Random Forest and XGBoost).
*Note*: These represent mathematical predictive association within the Cleveland cohort, NOT medical causation.


In [ ]:
feat_plot_path = os.path.join(artifacts_path, "feature_importance_plot.png")
if os.path.exists(feat_plot_path):
    img = Image.open(feat_plot_path)
    plt.figure(figsize=(18, 7))
    plt.imshow(img)
    plt.axis("off")
    plt.show()

feat_csv_path = os.path.join(artifacts_path, "feature_importance.csv")
df_feat = pd.read_csv(feat_csv_path)
print("Top 5 features for Logistic Regression (Robust L1):")
print(df_feat[df_feat["candidate_name"].str.contains("Logistic")].sort_values(by="abs_magnitude", ascending=False).head(5)[["feature", "importance_or_coef"]])


### Section 18: Limitations & Methodological Constraints
1. **Small Sample Size ($N=303$)**: Cross-validation estimates exhibit standard deviations of $\sim \pm 0.04$. Performance differences between candidates fall within mutual uncertainty bands.
2. **Single-Center Retrospective Cohort**: The Cleveland cohort reflects clinical practices and referral patterns from 1984. External transportability is unverified.
3. **Absence of Clinical Calibration**: Raw predicted probabilities are model-estimated screening scores and must NOT be interpreted as true physiological risk.
4. **Default Threshold Convention**: The 0.5 threshold was used strictly as an evaluation convention; clinical operating thresholds have not yet been established.


### Section 19: Phase 6 Handoff
Phase 5 experimentation is complete.

**Deliverables Ready for Next Phases**:
- `artifacts/heart/experiments/oof_predictions.csv`: 242 training observations with out-of-fold predicted probabilities.
- `artifacts/heart/experiments/test_predictions.csv`: 61 test observations with predicted probabilities.
- `artifacts/heart/experiments/candidate_summary.csv`: Summary of CV and hold-out test metrics.

**Handoff to Phase 6 (Threshold Analysis)**:
- Phase 6 will analyze candidate screening thresholds targeting high sensitivity ($\ge 90\%$) to minimize missed coronary artery disease cases.
- All subsequent calibration (Phase 7) and explainability (Phase 8) will follow sequentially.
